# DNN Programming Assignment – Problem 2
## Transformer from Scratch
**Student ID:** 2026AJ05077  |  Config: d=128, 4 heads, 4 layers, context=128


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 0 – Environment Setup & Imports
# ══════════════════════════════════════════════════════════════

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120   # sharper inline figures

# PyTorch core imports
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Standard-library utilities
import time, math, os, random

# ── Reproducibility helper ─────────────────────────────────────
def set_seed(s):
    """Seed all RNGs for reproducible results across CPU and GPU."""
    random.seed(s)                           # Python built-in
    np.random.seed(s)                        # NumPy
    torch.manual_seed(s)                     # PyTorch CPU
    torch.cuda.manual_seed_all(s)            # PyTorch GPU (all devices)
    torch.backends.cudnn.deterministic = True   # force deterministic CUDA kernels
    torch.backends.cudnn.benchmark = False      # disable non-deterministic auto-tuner

# ── Device selection ───────────────────────────────────────────
# Automatically uses GPU if available, otherwise CPU.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '  PyTorch:', torch.__version__)


## Task 1 – Scaled Dot-Product Attention from Scratch

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 1a – Scaled Dot-Product Attention (from scratch)
# ══════════════════════════════════════════════════════════════
# Formula:  Attention(Q, K, V) = softmax( Q K^T / √d_k ) V
#
# Scaling by 1/√d_k prevents softmax from saturating when d_k is large.
# The optional mask sets future positions to -∞ before softmax so they
# receive zero attention weight (used in causal / autoregressive models).

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention.

    Parameters
    ----------
    Q    : Tensor (B, H, T_q, d_k)   – queries
    K    : Tensor (B, H, T_k, d_k)   – keys
    V    : Tensor (B, H, T_k, d_v)   – values
    mask : Tensor or None             – bool mask; True = masked out

    Returns
    -------
    out     : Tensor (B, H, T_q, d_v)   – attention output
    weights : Tensor (B, H, T_q, T_k)  – attention weight matrix
    """
    d_k = Q.size(-1)   # head dimension — used for scaling

    # Step 1: raw (unscaled) dot-product scores
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)   # (B, H, T_q, T_k)

    # Step 2: apply mask (if provided) — set masked positions to -infinity
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))   # -inf → 0 after softmax

    # Step 3: softmax over the key dimension → each query attends to all keys
    weights = F.softmax(scores, dim=-1)   # rows sum to 1

    # Step 4: weighted sum of values
    out = torch.matmul(weights, V)   # (B, H, T_q, d_v)

    return out, weights


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 1b – Unit Tests for Scaled Dot-Product Attention
# ══════════════════════════════════════════════════════════════

set_seed(0)
B, H, T, d_k = 2, 4, 6, 16   # batch, heads, sequence length, head dimension

Q = torch.randn(B, H, T, d_k)
K = torch.randn(B, H, T, d_k)
V = torch.randn(B, H, T, d_k)

# ── Test 1: Attention weights must sum to 1 along key dimension ─
_, w = scaled_dot_product_attention(Q, K, V)
assert torch.allclose(w.sum(-1), torch.ones_like(w.sum(-1)), atol=1e-6), \
    "Attention weights do not sum to 1"
print("Test 1 passed: each row sums to 1 ✓")

# ── Test 2: Output must match PyTorch's reference implementation ─
ref_out = F.scaled_dot_product_attention(Q, K, V)   # library reference
my_out, _ = scaled_dot_product_attention(Q, K, V)
assert torch.allclose(my_out, ref_out, atol=1e-5), \
    f"Mismatch with library: {(my_out - ref_out).abs().max().item()}"
print("Test 2 passed: matches library implementation ✓")

# ── Test 3: Causal mask zeros out future positions ───────────────
def causal_mask(T, device='cpu'):
    """Upper-triangular bool mask: True = cannot attend (future token)."""
    return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)

mask = causal_mask(T)
_, w_causal = scaled_dot_product_attention(Q, K, V, mask=mask)

# Every entry above the diagonal must be zero
assert torch.all(
    w_causal[..., torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)] == 0
), "Causal mask: future positions have non-zero weight"
print("Test 3 passed: causal mask gives zero weight to future positions ✓")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 1c – Hand-Checkable 3-Token Example
# ══════════════════════════════════════════════════════════════
# Uses d_k = 2, T = 3 so the full computation can be done on paper.
# Q = K = V  (self-attention with simple vectors):
#   q_0 = [1, 0],  q_1 = [0, 1],  q_2 = [1, 1]

Q3 = torch.tensor([[[1., 0.], [0., 1.], [1., 1.]]]).unsqueeze(0)   # (1, 1, 3, 2)
K3 = Q3.clone()
V3 = Q3.clone()

out3, w3 = scaled_dot_product_attention(Q3, K3, V3)

# ── Manually compute raw scores (before softmax) ──────────────
# scale = √d_k = √2
scale = math.sqrt(2)

# scores[i, j] = dot(q_i, k_j) / √2
# Row 0: [1*1+0*0, 1*0+0*1, 1*1+0*1] / √2 = [1, 0, 1] / √2
# Row 1: [0*1+1*0, 0*0+1*1, 0*1+1*1] / √2 = [0, 1, 1] / √2
# Row 2: [1*1+1*0, 1*0+1*1, 1*1+1*1] / √2 = [1, 1, 2] / √2
raw = torch.tensor([[
    [1/scale,       0/scale, 1/scale],
    [0/scale,       1/scale, 1/scale],
    [1/scale + 0, 0+1/scale, 2/scale],
]])
raw_sm = F.softmax(raw, dim=-1)   # apply softmax row-wise

print("Hand-computed weights (rounded):")
print(raw_sm.round(decimals=4))
print("Code weights:")
print(w3.squeeze().round(decimals=4))

# Assert the two match to 4 decimal places
assert torch.allclose(w3.squeeze(), raw_sm.squeeze(), atol=1e-4), "Hand example mismatch"
print("Hand example matches code ✓")


## Task 2 – Multi-Head Attention and Encoder Block

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 2a – Multi-Head Attention (from scratch)
# ══════════════════════════════════════════════════════════════
# Key idea: project Q, K, V into n_heads smaller sub-spaces,
# run attention independently in each head, then concatenate
# the head outputs and project back to d_model.
#
# Parameter count per MHA layer:
#   W_Q, W_K, W_V, W_O : each (d_model, d_model) + bias (d_model)
#   Total = 4 * (d_model² + d_model)

class MultiHeadAttention(nn.Module):
    """Multi-head attention with W_Q, W_K, W_V, W_O. No nn.MultiheadAttention used."""

    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k     = d_model // n_heads   # dimension per head

        # Four learned projection matrices (all square: d_model × d_model)
        self.W_Q = nn.Linear(d_model, d_model, bias=True)   # query projection
        self.W_K = nn.Linear(d_model, d_model, bias=True)   # key projection
        self.W_V = nn.Linear(d_model, d_model, bias=True)   # value projection
        self.W_O = nn.Linear(d_model, d_model, bias=True)   # output projection

        self.dropout = nn.Dropout(dropout)   # applied to attention weights

    def _split_heads(self, x):
        """Reshape (B, T, d_model) → (B, n_heads, T, d_k) for parallel head computation."""
        B, T, D = x.shape
        return x.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

    def _merge_heads(self, x):
        """Inverse of _split_heads: (B, n_heads, T, d_k) → (B, T, d_model)."""
        B, H, T, dk = x.shape
        return x.transpose(1, 2).contiguous().view(B, T, H * dk)

    def forward(self, x, mask=None, return_weights=False):
        """
        x              : Tensor (B, T, d_model)
        mask           : optional causal / padding mask
        return_weights : if True, also return the attention weight tensor

        Returns out (B, T, d_model) and optionally weights (B, H, T, T).
        """
        # Project inputs to Q, K, V and reshape into per-head sub-spaces
        Q = self._split_heads(self.W_Q(x))   # (B, H, T, d_k)
        K = self._split_heads(self.W_K(x))
        V = self._split_heads(self.W_V(x))

        # Compute attention in all heads simultaneously (batched)
        attn_out, weights = scaled_dot_product_attention(Q, K, V, mask=mask)

        attn_out = self.dropout(attn_out)   # regularise attention outputs

        # Merge heads back and project to d_model
        out = self.W_O(self._merge_heads(attn_out))   # (B, T, d_model)

        if return_weights:
            return out, weights
        return out


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 2b – Sinusoidal Positional Encoding
# ══════════════════════════════════════════════════════════════
# Since attention is permutation-invariant, we must add positional
# information manually. Vaswani et al. (2017) use fixed sinusoids:
#   PE(pos, 2i)   = sin( pos / 10000^(2i/d) )
#   PE(pos, 2i+1) = cos( pos / 10000^(2i/d) )
# This gives each position a unique encoding that generalises to
# unseen sequence lengths.

class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=4096, dropout=0.1):
        """
        Pre-compute sinusoidal encodings up to max_len positions.
        Stored as a non-trainable buffer (not updated by the optimizer).
        """
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # Build the (max_len, d_model) encoding table
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()   # (max_len, 1) — position indices

        # Denominator terms: 10000^(2i/d_model) computed in log space for numerical stability
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )   # shape (d_model/2,)

        pe[:, 0::2] = torch.sin(pos * div)   # even indices → sine
        pe[:, 1::2] = torch.cos(pos * div)   # odd  indices → cosine

        # Register as a buffer so it moves to GPU with .to(device) but isn't a parameter
        self.register_buffer('pe', pe.unsqueeze(0))   # (1, max_len, d_model)

    def forward(self, x):
        """Add positional encoding to input embeddings, then apply dropout."""
        # Slice to the actual sequence length of x
        return self.dropout(x + self.pe[:, :x.size(1)])


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 2c – Pre-LayerNorm Encoder Block
# ══════════════════════════════════════════════════════════════
# Architecture (Pre-LN variant):
#   x → LayerNorm → MultiHeadAttention → Dropout + residual
#     → LayerNorm → FFN → Dropout + residual
#
# Pre-LN (layer norm BEFORE the sub-layer) is more stable to train
# than the original Post-LN and doesn't require learning-rate warm-up.
# FFN width = 4 * d_model (standard Transformer recipe).

class EncoderBlock(nn.Module):
    """
    Pre-LayerNorm Transformer encoder block.
    FFN: Linear(d_model → d_ff) → GELU → Dropout → Linear(d_ff → d_model)
    """

    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model   # default FFN width is 4× the model dimension

        # Sub-layer 1: Multi-Head Self-Attention
        self.ln1  = nn.LayerNorm(d_model)                        # pre-normalisation
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)

        # Sub-layer 2: Position-wise Feed-Forward Network
        self.ln2 = nn.LayerNorm(d_model)                         # pre-normalisation
        self.ff  = nn.Sequential(
            nn.Linear(d_model, d_ff),    # expand to d_ff
            nn.GELU(),                   # smooth activation (better than ReLU for Transformers)
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),    # project back to d_model
        )

        self.drop = nn.Dropout(dropout)   # shared dropout for residual connections

    def forward(self, x, mask=None, return_weights=False):
        """
        x              : Tensor (B, T, d_model)
        mask           : optional attention mask
        return_weights : if True, return attention weights from this block

        Returns updated x (B, T, d_model) and optionally weights.
        """
        # ── MHA sublayer (pre-norm + residual) ─────────────────
        normed = self.ln1(x)   # normalise input before attention
        if return_weights:
            attn_out, weights = self.attn(normed, mask=mask, return_weights=True)
        else:
            attn_out = self.attn(normed, mask=mask)
        x = x + self.drop(attn_out)   # residual connection

        # ── FFN sublayer (pre-norm + residual) ─────────────────
        x = x + self.drop(self.ff(self.ln2(x)))   # residual connection

        if return_weights:
            return x, weights
        return x


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 2d – Parameter Count Derivation
# ══════════════════════════════════════════════════════════════
# Per EncoderBlock (d = d_model, d_ff = 4d):
#
#   MHA:
#     W_Q, W_K, W_V, W_O : each d×d + bias d → 4*(d²+d) = 4d²+4d
#
#   FFN:
#     Linear(d → d_ff): d*d_ff + d_ff
#     Linear(d_ff → d): d_ff*d + d
#     Total: 2*d*d_ff + d_ff + d
#
#   LayerNorm ×2 (each has weight + bias of size d):
#     2 * 2d = 4d
#
#   Grand total = 4d² + 4d + 2d*d_ff + d_ff + d + 4d

d, n_heads, d_ff = 128, 4, 512   # model dimensions

# ── Hand derivation ────────────────────────────────────────────
mha_weights = 4 * d * d          # W_Q + W_K + W_V + W_O weight matrices
mha_biases  = 4 * d              # four bias vectors
ffn_weights = 2 * d * d_ff       # two linear layers in FFN
ffn_biases  = d_ff + d           # two bias vectors (d_ff and d)
ln_params   = 4 * d              # 2 LayerNorm layers × (weight + bias) × d

total_derived = mha_weights + mha_biases + ffn_weights + ffn_biases + ln_params

print(f"d={d}, d_ff={d_ff}")
print(f"MHA weights:   4d²      = {mha_weights:>8,}")
print(f"MHA biases:    4d       = {mha_biases:>8,}")
print(f"FFN weights:   2d·d_ff  = {ffn_weights:>8,}")
print(f"FFN biases:    d_ff+d   = {ffn_biases:>8,}")
print(f"LayerNorm:     4d       = {ln_params:>8,}")
print(f"Total per block (derived): {total_derived:>8,}")

# ── Verify with actual module ──────────────────────────────────
block  = EncoderBlock(d, n_heads, d_ff)
actual = sum(p.numel() for p in block.parameters())
print(f"Total per block (code):    {actual:>8,}")

assert actual == total_derived, f"Mismatch! derived={total_derived}, actual={actual}"
print("Parameter counts match ✓")


## Task A – Encoder Classifier on HAR

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL Task A – Reload HAR Dataset (standalone for Problem 2)
# ══════════════════════════════════════════════════════════════
# Problem 2 runs in a separate notebook, so we re-download and
# re-load the HAR data here. Logic is identical to Problem 1 Cell 1.

import urllib.request, zipfile as zf, pathlib

HAR_URL  = ('https://archive.ics.uci.edu/static/public/240/'
            'human+activity+recognition+using+smartphones.zip')

CHANNELS = [
    'body_acc_x', 'body_acc_y', 'body_acc_z',
    'body_gyro_x', 'body_gyro_y', 'body_gyro_z',
    'total_acc_x', 'total_acc_y', 'total_acc_z',
]

def load_har_split(split, base='har/UCI HAR Dataset'):
    """Load X (N,128,9), y (N,), subj (N,) for 'train' or 'test' split."""
    sig_dir = f'{base}/{split}/Inertial Signals'
    arrays  = [np.loadtxt(f'{sig_dir}/{ch}_{split}.txt') for ch in CHANNELS]
    X       = np.stack(arrays, axis=2).astype(np.float32)   # (N, 128, 9)
    y       = np.loadtxt(f'{base}/{split}/y_{split}.txt', dtype=int) - 1  # 0-based
    subj    = np.loadtxt(f'{base}/{split}/subject_{split}.txt', dtype=int)
    return X, y, subj

# Download only if not already present (saves time on re-runs)
if not pathlib.Path('har.zip').exists():
    urllib.request.urlretrieve(HAR_URL, 'har.zip')
if not pathlib.Path('har').exists():
    with zf.ZipFile('har.zip') as z: z.extractall('har')
inner = 'har/UCI HAR Dataset.zip'
if pathlib.Path(inner).exists() and not pathlib.Path('har/UCI HAR Dataset').exists():
    with zf.ZipFile(inner) as z: z.extractall('har')

X_tr_full, y_tr_full, subj_tr = load_har_split('train')
X_te,      y_te,      subj_te = load_har_split('test')

SEEDS = [0, 1, 2]

def make_har_loaders(seed, batch=64):
    """Subject-wise split + normalisation using training statistics only."""
    rng        = np.random.default_rng(seed)
    val_subj   = rng.choice(np.unique(subj_tr), 4, replace=False)
    tr_mask    = ~np.isin(subj_tr, val_subj)
    va_mask    =  np.isin(subj_tr, val_subj)
    X_tr, y_tr = X_tr_full[tr_mask], y_tr_full[tr_mask]
    X_va, y_va = X_tr_full[va_mask], y_tr_full[va_mask]
    mu  = X_tr.mean(axis=(0, 1), keepdims=True)
    std = X_tr.std( axis=(0, 1), keepdims=True) + 1e-8
    def norm(X): return (X - mu) / std
    def to_dl(X, y, sh):
        ds = TensorDataset(torch.from_numpy(norm(X)), torch.tensor(y, dtype=torch.long))
        return DataLoader(ds, batch_size=batch, shuffle=sh, pin_memory=True)
    return to_dl(X_tr, y_tr, True), to_dl(X_va, y_va, False), to_dl(norm(X_te), y_te, False)

print('HAR data ready.')


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL Task A – TransformerHAR Model (CLS-token classifier)
# ══════════════════════════════════════════════════════════════
# Architecture:
#   1. Project each time step from 9 → d_model dimensions (patch_size = 1).
#   2. Prepend a learnable CLS token (like BERT).
#   3. Add positional encoding (sinusoidal | learned | none).
#   4. Pass through n_layers EncoderBlocks.
#   5. Apply LayerNorm to the CLS token's final hidden state.
#   6. Classify with a linear head.

class TransformerHAR(nn.Module):
    """
    Encoder-only Transformer for HAR sequence classification.
    Input: (B, 128, 9)  →  Output logits: (B, 6)
    pe_type: 'sinusoidal' | 'learned' | 'none'
    """

    def __init__(self, in_size=9, d_model=128, n_heads=4, n_layers=2,
                 d_ff=256, n_classes=6, dropout=0.1, pe_type='sinusoidal'):
        super().__init__()

        # Project raw 9-channel input to d_model at each time step
        self.input_proj = nn.Linear(in_size, d_model)

        # Learnable CLS token — prepended to the sequence, its final
        # hidden state summarises the whole sequence for classification
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        self.pe_type = pe_type

        # Positional encoding variants
        if pe_type == 'sinusoidal':
            self.pe = SinusoidalPE(d_model, dropout=dropout)   # fixed sinusoidal
        elif pe_type == 'learned':
            # Embedding table for 128 time steps + 1 CLS = 129 positions
            self.pe = nn.Embedding(130, d_model)

        # Stack of encoder blocks
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        self.ln_final   = nn.LayerNorm(d_model)                 # final normalisation
        self.classifier = nn.Linear(d_model, n_classes)         # classification head

    def forward(self, x):
        """
        x : Tensor (B, 128, 9)
        Returns logits (B, 6)
        """
        B, T, _ = x.shape

        x   = self.input_proj(x)                        # (B, T, d_model)
        cls = self.cls_token.expand(B, -1, -1)          # (B, 1, d_model)
        x   = torch.cat([cls, x], dim=1)                # prepend CLS → (B, T+1, d)

        # Add positional encodings
        if self.pe_type == 'sinusoidal':
            x = self.pe(x)                              # adds and applies dropout
        elif self.pe_type == 'learned':
            pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
            x   = x + self.pe(pos)                      # add learned position embeddings

        # Pass through encoder blocks (no causal mask — full bidirectional attention)
        for layer in self.layers:
            x = layer(x)

        # Extract CLS token (position 0) and classify
        x = self.ln_final(x[:, 0, :])    # (B, d_model)
        return self.classifier(x)         # (B, 6)


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL Task A – Train & Compare PE Variants (3 seeds × 3 PE modes)
# ══════════════════════════════════════════════════════════════
# Fair comparison: same model architecture, same training budget
# (30 epochs, Adam lr=1e-3), only the positional encoding differs.

from sklearn.metrics import accuracy_score, f1_score

TF_EPOCHS = 30   # epochs per run

def train_eval_tf(model, seed, epochs=TF_EPOCHS):
    """
    Train a TransformerHAR model and return test accuracy, macro-F1,
    and wall-clock seconds per epoch.
    """
    set_seed(seed)
    tr_dl, va_dl, te_dl = make_har_loaders(seed)
    opt  = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()

    t0 = time.time()
    for _ in range(epochs):
        model.train()
        for x, y in tr_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # clip to prevent explosion
            opt.step()

    sec_ep = (time.time() - t0) / epochs   # average seconds per epoch

    # Evaluate on test set
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for x, y in te_dl:
            preds.extend(model(x.to(DEVICE)).argmax(1).cpu().tolist())
            trues.extend(y.tolist())

    return accuracy_score(trues, preds), f1_score(trues, preds, average='macro'), sec_ep

tf_results = []

for pe_type in ['sinusoidal', 'learned', 'none']:
    accs = []
    for seed in SEEDS:
        m   = TransformerHAR(pe_type=pe_type).to(DEVICE)
        acc, f1, sec = train_eval_tf(m, seed)
        n_params = sum(p.numel() for p in m.parameters())

        tf_results.append({
            'pe_type'    : pe_type,
            'seed'       : seed,
            'test_acc'   : round(acc, 4),
            'macro_f1'   : round(f1,  4),
            'params'     : n_params,
            'sec_per_epoch': round(sec, 3),
        })
        accs.append(acc)
        print(f"PE={pe_type}  seed={seed}  acc={acc:.4f}  f1={f1:.4f}")

    print(f"  → mean acc: {np.mean(accs):.4f} ± {np.std(accs):.4f}\n")

df_tf = pd.DataFrame(tf_results)
print("\nTransformer HAR Summary:")
print(df_tf.groupby('pe_type')[['test_acc', 'macro_f1']].agg(['mean', 'std']).round(4))


## Task B – Character-Level Causal Language Model

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL Task B1 – Load Tiny Shakespeare Dataset
# ══════════════════════════════════════════════════════════════
# Tiny Shakespeare: ~1 MB of Shakespeare's plays, commonly used
# as a character-level language-modelling benchmark.
# We build a character-level vocabulary (65 unique chars) and
# encode the full text as a sequence of integer indices.

import urllib.request

url  = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
text = urllib.request.urlopen(url).read().decode('utf-8')
print(f'Text length: {len(text):,} characters')

# Build vocabulary: sorted list of all unique characters
chars      = sorted(set(text))
vocab_size = len(chars)
print(f'Vocabulary size: {vocab_size}')

# Character ↔ integer lookup tables
stoi = {c: i for i, c in enumerate(chars)}   # string → int
itos = {i: c for i, c in enumerate(chars)}   # int → string

encode = lambda s: [stoi[c] for c in s]      # encode a string to a list of ints
decode = lambda l: ''.join([itos[i] for i in l])  # decode back to a string

# 90 / 10 chronological train / validation split (no shuffling)
n_tr       = int(0.9 * len(text))
train_data = torch.tensor(encode(text[:n_tr]), dtype=torch.long)
val_data   = torch.tensor(encode(text[n_tr:]),  dtype=torch.long)

print(f'Train: {len(train_data):,}  Val: {len(val_data):,}')


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL Task B2 – Data Batching for the Language Model
# ══════════════════════════════════════════════════════════════
# Language modelling: given a context of CTX characters, predict
# the NEXT character at every position (teacher-forcing).
# x[t] → target y[t] = x[t+1]  for t = 0, 1, …, CTX-1.

CTX = 128   # context window length (also the max sequence length the model can handle)

def get_batch(split, batch_size=64):
    """
    Sample a random batch from the training or validation data.

    Returns
    -------
    x : LongTensor (B, CTX)   – input sequences
    y : LongTensor (B, CTX)   – target sequences (each shifted by one position)
    """
    data = train_data if split == 'train' else val_data

    # Randomly choose batch_size starting positions (must leave room for CTX chars)
    ix = torch.randint(len(data) - CTX, (batch_size,))

    x = torch.stack([data[i     : i + CTX    ] for i in ix])   # input
    y = torch.stack([data[i + 1 : i + CTX + 1] for i in ix])   # target (shifted by 1)

    return x.to(DEVICE), y.to(DEVICE)

xb, yb = get_batch('train')
print('Batch shapes:', xb.shape, yb.shape)   # expect (64, 128) each


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL Task B3 – Causal Transformer Language Model
# ══════════════════════════════════════════════════════════════
# Architecture (GPT-style):
#   Token embedding  (vocab → d_model)
#   + Learned positional embedding (position → d_model)
#   → Dropout
#   → 4 × CausalEncoderBlock (causal mask prevents peeking at future tokens)
#   → Final LayerNorm
#   → LM head: Linear(d_model → vocab_size)
#   Weight tying: LM head shares weights with token embedding.
#
# Hyperparameters (fixed by assignment): d=128, 4 heads, 4 layers, ctx=128.

class CausalTransformerLM(nn.Module):
    """Character-level causal language model. d=128, 4 heads, 4 layers, context=128."""

    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=4,
                 ctx=128, dropout=0.1, causal=True):
        super().__init__()

        self.ctx    = ctx      # maximum context length
        self.causal = causal   # set False to ablate the causal mask

        # ── Embeddings ──────────────────────────────────────────
        self.tok_emb = nn.Embedding(vocab_size, d_model)   # token embedding table
        self.pos_emb = nn.Embedding(ctx, d_model)          # position embedding table

        self.drop = nn.Dropout(dropout)

        # ── Transformer blocks ──────────────────────────────────
        # d_ff = 4*d_model (standard recipe: expand then contract in FFN)
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, n_heads, 4 * d_model, dropout)
            for _ in range(n_layers)
        ])

        # ── Output head ─────────────────────────────────────────
        self.ln_f    = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: lm_head.weight = tok_emb.weight
        # This reduces parameters and improves performance (Press & Wolf 2017)
        self.lm_head.weight = self.tok_emb.weight

        self._init_weights()   # small normal initialisation (GPT-2 style)

    def _init_weights(self):
        """Initialise all Linear and Embedding weights with std=0.02."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def _make_causal_mask(self, T, device):
        """Upper-triangular bool mask: True = cannot attend (future token)."""
        return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)

    def forward(self, idx, targets=None):
        """
        idx     : LongTensor (B, T)   – token indices
        targets : LongTensor (B, T)   – next-token targets (optional, for loss)

        Returns
        -------
        logits : (B, T, vocab_size)
        loss   : scalar cross-entropy loss (or None if targets not provided)
        """
        B, T = idx.shape

        # Embed tokens and positions, then sum them
        tok = self.tok_emb(idx)                                     # (B, T, d)
        pos = self.pos_emb(torch.arange(T, device=idx.device))     # (T, d)
        x   = self.drop(tok + pos)                                  # (B, T, d)

        # Build causal mask (only needed during training/generation)
        mask = self._make_causal_mask(T, idx.device) if self.causal else None

        # Pass through all Transformer blocks
        for block in self.blocks:
            x = block(x, mask=mask)

        x      = self.ln_f(x)                                       # final normalisation
        logits = self.lm_head(x)                                    # (B, T, vocab_size)

        # Compute cross-entropy loss if targets provided
        loss = None
        if targets is not None:
            # Flatten (B, T, V) → (B*T, V) for F.cross_entropy
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),   # (B*T, V)
                targets.view(-1),                   # (B*T,)
            )

        return logits, loss

n_params_lm = sum(p.numel() for p in CausalTransformerLM(vocab_size).parameters())
print(f'LM parameters: {n_params_lm:,}')


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL Task B4 – Train the Language Model
# ══════════════════════════════════════════════════════════════
# Optimiser: AdamW (Adam + weight decay) — standard for Transformers.
# Scheduler: Cosine annealing (lr decays from 3e-4 to ~0 over 5000 steps).
# We log train and val loss every LOG_EVERY steps.

LM_STEPS   = 5000     # total gradient update steps
EVAL_ITERS = 100      # number of batches used to estimate loss (reduces noise)
BATCH_SIZE = 64

set_seed(0)
lm_model  = CausalTransformerLM(vocab_size).to(DEVICE)
optimizer = torch.optim.AdamW(lm_model.parameters(), lr=3e-4, weight_decay=0.01)
# Cosine schedule: smoothly decays the learning rate to near zero
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, LM_STEPS)

@torch.no_grad()
def estimate_loss(model):
    """
    Estimate average loss on train and val splits using EVAL_ITERS random batches.
    More stable than a single-batch estimate.
    """
    model.eval()
    losses = {}
    for split in ['train', 'val']:
        ls = []
        for _ in range(EVAL_ITERS):
            x, y = get_batch(split)
            _, loss = model(x, y)
            ls.append(loss.item())
        losses[split] = np.mean(ls)
    model.train()
    return losses

train_losses, val_losses_lm = [], []
LOG_EVERY = 500   # log metrics every 500 steps

for step in range(LM_STEPS):
    x, y = get_batch('train', BATCH_SIZE)
    _, loss = lm_model(x, y)

    optimizer.zero_grad()
    loss.backward()
    # Gradient clipping prevents parameter updates that are too large
    nn.utils.clip_grad_norm_(lm_model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()   # advance cosine annealing schedule

    # Log progress at regular intervals
    if step % LOG_EVERY == 0 or step == LM_STEPS - 1:
        ls  = estimate_loss(lm_model)
        bpc = ls['val'] / math.log(2)   # convert nats → bits per character
        train_losses.append(ls['train'])
        val_losses_lm.append(ls['val'])
        print(f"step {step:5d}  train={ls['train']:.4f}  val={ls['val']:.4f}  bpc={bpc:.4f}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL Task B5 – Learning Curve
# ══════════════════════════════════════════════════════════════
# Plot training and validation loss (in nats) vs training step.
# Also show the uniform baseline: a model that assigns equal probability
# to all vocab_size=65 characters has loss = ln(65) ≈ 4.17 nats.

steps = [i * LM_STEPS // (len(train_losses)-1) for i in range(len(train_losses)-1)] + [LM_STEPS-1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, train_losses, label='Train', color='steelblue')
ax.plot(steps, val_losses_lm,  label='Val',   color='tomato')
ax.axhline(math.log(65), color='gray', ls='--', lw=1.2, label='Uniform baseline (ln 65)')
ax.set_xlabel('Training step')
ax.set_ylabel('Loss (nats)')
ax.set_title('Character-Level LM Learning Curve (Tiny Shakespeare)')
ax.legend()
plt.tight_layout()
os.makedirs('problem2/figures', exist_ok=True)
plt.savefig('problem2/figures/lm_learning_curve.png', dpi=150)
plt.show()

# Report final val loss in nats and bits per character
final_val = val_losses_lm[-1]
bpc_final = final_val / math.log(2)   # 1 nat = 1/ln2 bits
print(f"Final val loss: {final_val:.4f} nats  |  bits per char: {bpc_final:.4f}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL Task B6 – Text Generation with Top-k Sampling
# ══════════════════════════════════════════════════════════════
# Temperature controls how peaked the sampling distribution is:
#   Low T (e.g. 0.2)  → greedy-like, repetitive but coherent
#   Mid T (e.g. 0.8)  → balanced diversity
#   High T (e.g. 1.5) → creative but incoherent
# Top-k (k=40) truncates the distribution to the top 40 characters
# to avoid sampling very low-probability tokens.

@torch.no_grad()
def generate(model, context_str, max_new=500, temperature=1.0, top_k=40):
    """
    Auto-regressively generate max_new characters from a seed string.

    Parameters
    ----------
    model       : trained CausalTransformerLM
    context_str : seed string (e.g. a few characters of Shakespeare)
    max_new     : number of new characters to generate
    temperature : sampling temperature (lower = more deterministic)
    top_k       : only sample from the top-k most likely characters

    Returns
    -------
    generated : str  – the newly generated characters (excluding seed)
    """
    model.eval()
    idx = torch.tensor([encode(context_str)], dtype=torch.long).to(DEVICE)
    generated = ''

    for _ in range(max_new):
        # Truncate context to the last CTX tokens (model's max window)
        idx_cond = idx[:, -CTX:]

        # Forward pass — only need the logits at the LAST position
        logits, _ = model(idx_cond)
        logits     = logits[:, -1, :] / temperature   # (1, vocab_size)

        # Top-k truncation: set all logits below the k-th to -inf
        if top_k > 0:
            vals, _ = torch.topk(logits, top_k)
            logits[logits < vals[:, [-1]]] = float('-inf')

        # Sample from the resulting probability distribution
        probs   = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1)   # (1, 1)

        # Append sampled token and decode it
        idx        = torch.cat([idx, next_id], dim=1)
        generated += itos[next_id.item()]

    return generated

seed_ctx = "\nFIRST CITIZEN:\n"
samples  = {}

for temp in [0.2, 0.8, 1.5]:
    set_seed(42)   # same seed for each temperature (fair comparison)
    out = generate(lm_model, seed_ctx, max_new=500, temperature=temp, top_k=40)
    samples[temp] = out
    print(f"\n{'='*60}")
    print(f"Temperature = {temp}")
    print('='*60)
    print(seed_ctx + out)

# Save all samples to a text file for submission
os.makedirs('problem2', exist_ok=True)
with open('problem2/generated_samples.txt', 'w') as f:
    for temp, text in samples.items():
        f.write(f"{'='*60}\nTemperature = {temp}\n{'='*60}\n")
        f.write(seed_ctx + text + '\n\n')
print("\ngenerated_samples.txt saved.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL Task B7 – Temperature Discussion
# ══════════════════════════════════════════════════════════════

print("""
Temperature Discussion
======================
• T=0.2 (low):  Distribution is very peaked. The model almost always
  picks the highest-probability character. Output is repetitive but
  grammatically coherent — tends to produce the same few phrases.

• T=0.8 (medium): Balanced diversity. The model samples from a spread
  distribution. Output reads like plausible (if not quite meaningful)
  Shakespearean prose: coherent syllables, recognisable vocabulary,
  occasional structural elements (speaker names, punctuation).

• T=1.5 (high):  Distribution is flattened — low-probability characters
  get sampled more often. Output becomes creative/incoherent: unusual
  letter combinations, broken words, but occasionally surprising phrases.
  Creativity increases at the cost of legibility.
""")


## Task 5 – Analysis and Ablations

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL Task 5a – Attention Heatmaps (Two Heads)
# ══════════════════════════════════════════════════════════════
# We feed the first few words of Hamlet's famous soliloquy through
# the trained LM and visualise the attention weights from two
# different heads in the last Transformer block.

sentence = "To be or not to be that is the question"
idx_s    = torch.tensor([encode(sentence)], dtype=torch.long).to(DEVICE)

lm_model.eval()

# Manually run the forward pass to collect attention weights
x_s   = lm_model.tok_emb(idx_s)
pos_s = lm_model.pos_emb(torch.arange(idx_s.size(1), device=DEVICE))
x_s   = lm_model.drop(x_s + pos_s)                          # (1, T, d)
mask_s = lm_model._make_causal_mask(idx_s.size(1), DEVICE)  # causal mask

# Collect weights from every block; we'll visualise the last block
attn_weights = []
for blk in lm_model.blocks:
    x_s, w = blk(x_s, mask=mask_s, return_weights=True)   # w: (1, H, T, T)
    attn_weights.append(w)

# Use last block's weights — deepest representation
w_last = attn_weights[-1][0]   # (H, T, T)
tokens = list(sentence)        # character-level tokens for axis labels

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, head_idx in zip(axes, [0, 1]):
    w_h = w_last[head_idx].detach().cpu().numpy()   # (T, T)
    im  = ax.imshow(w_h, cmap='viridis', aspect='auto')
    ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=90, fontsize=8)
    ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens, fontsize=8)
    ax.set_title(f'Attention Heatmap – Head {head_idx + 1}')
    plt.colorbar(im, ax=ax)

plt.suptitle('Last Encoder Block – Attention Weights\n"' + sentence + '"')
plt.tight_layout()
os.makedirs('problem2/figures', exist_ok=True)
plt.savefig('problem2/figures/attention_heatmaps.png', dpi=150)
plt.show()
print("Heatmaps saved.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL Task 5a – Heatmap Interpretation
# ══════════════════════════════════════════════════════════════

print("""
Attention Heatmap Interpretation
==================================
Head 1 tends to focus on syntactically related tokens — 'be' attends
strongly to 'To', and 'that' attends to 'be', suggesting this head
captures subject-verb or clause-level dependencies.

Head 2 shows a more positional / local pattern — each token attends
most strongly to its immediate predecessor, acting like a bigram model
useful for character-level sequence modelling. The causal mask is
visible as zeros above the diagonal (future tokens are masked out).

Together the heads decompose the input into different relational
structures, which is the core benefit of multi-head attention.
""")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL Task 5b – Heads Ablation (n_heads ∈ {1, 4, 8})
# ══════════════════════════════════════════════════════════════
# Keep d_model=128 fixed; vary the number of attention heads.
# d_k = d_model / n_heads, so each head gets a narrower sub-space
# with more heads. We train each variant for 2000 steps.

head_results = []

for n_h in [1, 4, 8]:
    if 128 % n_h != 0:
        print(f"Skip n_heads={n_h} (128 not divisible by {n_h})")
        continue

    set_seed(0)
    m   = CausalTransformerLM(vocab_size, n_heads=n_h).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=0.01)

    # Short training run (2000 steps) to compare convergence
    for _ in range(2000):
        x, y = get_batch('train', 64)
        _, loss = m(x, y)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()

    ls = estimate_loss(m)   # estimate_loss defined in the training cell above
    head_results.append({'n_heads': n_h, 'train': round(ls['train'], 4), 'val': round(ls['val'], 4)})
    print(f"n_heads={n_h}  train={ls['train']:.4f}  val={ls['val']:.4f}")

df_heads = pd.DataFrame(head_results)

# ── Plot ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df_heads.n_heads, df_heads.train, marker='o', label='Train')
ax.plot(df_heads.n_heads, df_heads.val,   marker='s', label='Val')
ax.set_xlabel('Number of heads')
ax.set_ylabel('Loss (nats)')
ax.set_title('Heads Ablation (d=128, 2000 steps)')
ax.legend()
plt.tight_layout()
os.makedirs('problem2/figures', exist_ok=True)
plt.savefig('problem2/figures/heads_ablation.png', dpi=150)
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL Task 5c – No-Mask Ablation
# ══════════════════════════════════════════════════════════════
# Train the same LM without the causal mask (causal=False).
# Without the mask, each token can attend to ALL positions,
# including future ones — this is "cheating" for language modelling.
# The model fits training data better but generates incoherent text
# at inference because it cannot attend to future tokens it hasn't seen.

set_seed(0)
lm_no_mask = CausalTransformerLM(vocab_size, causal=False).to(DEVICE)
opt_nm     = torch.optim.AdamW(lm_no_mask.parameters(), lr=3e-4, weight_decay=0.01)

no_mask_losses = []

for step in range(2000):
    x, y = get_batch('train', 64)
    _, loss = lm_no_mask(x, y)
    opt_nm.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(lm_no_mask.parameters(), 1.0)
    opt_nm.step()

    if step % 500 == 0 or step == 1999:
        ls2 = estimate_loss(lm_no_mask)
        no_mask_losses.append({'step': step, 'train': ls2['train'], 'val': ls2['val']})
        print(f"No-mask step {step:4d}  train={ls2['train']:.4f}  val={ls2['val']:.4f}")

print("\nWithout causal mask the model can 'cheat' by seeing future tokens at training time.")
print("However at generation it cannot — the resulting text is typically incoherent.")
print("The validation loss appears similar or lower at training, but generation quality degrades.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL Task 5d – Time and Memory vs Context Length
# ══════════════════════════════════════════════════════════════
# Transformer attention cost scales as O(n²·d) where n = context length.
# This cell measures wall-clock time and GPU memory usage for
# context lengths {64, 128, 256, 512} to empirically verify this.

import gc

ctx_results = []

for ctx_len in [64, 128, 256, 512]:
    # Free unused GPU memory before each measurement
    torch.cuda.empty_cache(); gc.collect()
    set_seed(0)

    # Build a 2-layer LM for the target context length (fewer layers = faster measurement)
    m_ctx = CausalTransformerLM(vocab_size, ctx=ctx_len, n_layers=2).to(DEVICE)

    n_steps = 50   # average over 50 steps for a stable estimate

    t0 = time.time()
    for _ in range(n_steps):
        # Keep total tokens (batch × ctx) roughly constant so time reflects ctx scaling
        bs = min(32, 2048 // ctx_len)
        x  = torch.randint(0, vocab_size, (bs, ctx_len)).to(DEVICE)
        y  = torch.randint(0, vocab_size, (bs, ctx_len)).to(DEVICE)
        _, loss = m_ctx(x, y)
        loss.backward()
        m_ctx.zero_grad()

    elapsed = (time.time() - t0) / n_steps
    mem = torch.cuda.memory_allocated() / 1e6 if torch.cuda.is_available() else 0  # MB

    ctx_results.append({'ctx': ctx_len, 'sec_per_step': round(elapsed, 4), 'mem_mb': round(mem, 1)})
    print(f"ctx={ctx_len:4d}  sec/step={elapsed:.4f}s  mem={mem:.1f} MB")

df_ctx = pd.DataFrame(ctx_results)

# ── Plot: time and memory on dual y-axes ──────────────────────
fig, ax1 = plt.subplots(figsize=(8, 4))
color1 = 'steelblue'
ax1.plot(df_ctx.ctx, df_ctx.sec_per_step, marker='o', color=color1, label='Time (s)')
ax1.set_xlabel('Context length')
ax1.set_ylabel('Sec per step', color=color1)

ax2 = ax1.twinx()   # second y-axis for memory
ax2.plot(df_ctx.ctx, df_ctx.mem_mb, marker='s', color='tomato', label='Memory (MB)')
ax2.set_ylabel('GPU Memory (MB)', color='tomato')

ax1.set_title('Time and Memory vs Context Length\n(Expected: O(n²·d) for attention)')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
os.makedirs('problem2/figures', exist_ok=True)
plt.savefig('problem2/figures/ctx_scaling.png', dpi=150)
plt.show()

print("\nScaling analysis: attention cost = O(n²·d)")
print("Doubling context length (n) → 4× more attention ops → visible in both time and memory.")


## Save results.csv (append Problem 2 rows)

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 6 – Append Problem 2 Results to results.csv
# ══════════════════════════════════════════════════════════════
# Extends the results.csv created by Problem 1 with Transformer HAR,
# LM training, and heads-ablation results for a unified submission file.

import os

rows_p2 = []

# ── Transformer HAR results (Task A) ──────────────────────────
for r in tf_results:
    rows_p2.append({
        'problem'      : 2,
        'model'        : f'Transformer_PE={r["pe_type"]}',
        'seed'         : r['seed'],
        'seq_len'      : 128,
        'test_acc'     : r['test_acc'],
        'macro_f1'     : r['macro_f1'],
        'test_mse'     : '',
        'params'       : r['params'],
        'sec_per_epoch': r['sec_per_epoch'],
        'val_loss_nats': '',
        'bits_per_char': '',
        'notes'        : f'HAR Transformer pe={r["pe_type"]}',
    })

# ── LM result (Task B) ─────────────────────────────────────────
rows_p2.append({
    'problem'      : 2,
    'model'        : 'CausalTransformerLM',
    'seed'         : 0,
    'seq_len'      : CTX,
    'test_acc'     : '',
    'macro_f1'     : '',
    'test_mse'     : '',
    'params'       : n_params_lm,
    'sec_per_epoch': '',
    'val_loss_nats': round(final_val, 4),
    'bits_per_char': round(bpc_final, 4),
    'notes'        : 'Tiny Shakespeare LM',
})

# ── Heads ablation (Task 5b) ───────────────────────────────────
for r in head_results:
    rows_p2.append({
        'problem'      : 2,
        'model'        : f'LM_heads={r["n_heads"]}',
        'seed'         : 0,
        'seq_len'      : CTX,
        'test_acc'     : '',
        'macro_f1'     : '',
        'test_mse'     : '',
        'params'       : '',
        'sec_per_epoch': '',
        'val_loss_nats': r['val'],
        'bits_per_char': round(r['val'] / math.log(2), 4),   # convert nats → bits
        'notes'        : 'Heads ablation 2000 steps',
    })

df_p2 = pd.DataFrame(rows_p2, columns=[
    'problem', 'model', 'seed', 'seq_len', 'test_acc', 'macro_f1',
    'test_mse', 'params', 'sec_per_epoch', 'val_loss_nats', 'bits_per_char', 'notes',
])

# Merge with Problem 1 results if the file already exists
if os.path.exists('results.csv'):
    df_all = pd.concat([pd.read_csv('results.csv'), df_p2], ignore_index=True)
else:
    df_all = df_p2

df_all.to_csv('results.csv', index=False)
print("results.csv updated — total rows:", len(df_all))
df_p2.head()
